# Capstone Session 2: Data Processing and Statistical Analysis

Objective: To finalize data processing by de-normalizing coded variables (age and income), calculate key statistical metrics, compare manual vs. automated analysis, and identify columns requiring further encoding.

## 1. Initial Cleanup and Intermediate Export (Task 1: Impute Missing Income)
This script loads the raw data, handles the missing values in the income column (which is a necessary step before scaling), and generates the intermediate file NSMES1988new.csv as referenced in the Session 2 requirements.

In [15]:
import pandas as pd
import numpy as np # Adding numpy, which is often imported alongside pandas for numerical ops

print("--- Starting 2a: Initial Cleanup (Impute Missing Values) ---")

input_file = 'NSMES1988.csv'
output_file = 'NSMES1988new.csv'

# --- 1. Load Data ---
try:
    # Use the first column as the index (index_col=0) to prevent it from becoming an 
    # 'Unnamed: 0' column in the DataFrame.
    df = pd.read_csv(input_file, index_col=0)
    print(f"Loaded '{input_file}'. Shape: {df.shape}")
except FileNotFoundError:
    # Essential error handling for file existence
    print(f"Error: Could not find '{input_file}'. Please ensure it is in the same directory.")
    # In a real environment, you might log an error and stop execution gracefully.
    exit()

# --- 2. Data Cleaning: Impute Missing Income (Prerequisite for Scaling) ---

# Check for missing values in the target column
missing_count = df['income'].isnull().sum()
print(f"Detected {missing_count} missing values in 'income'.")

# Impute the median to avoid bias from potential outliers, as income data is often skewed.
median_income = df['income'].median()

# Fill the missing values with the calculated median
# We explicitly re-assign the result for modern Pandas versions.
df['income'] = df['income'].fillna(median_income)
print(f"Missing income values imputed with the Median: {median_income:.4f}")

# Verify cleaning
print(f"Missing values remaining in 'income': {df['income'].isnull().sum()}")

# --- 3. Export Intermediate File ---
# Save the DataFrame with the imputed income data for the next session.
df.to_csv(output_file, index=True)
print(f"\nSuccessfully saved the cleaned (but NOT scaled/transformed) data to '{output_file}'.")

# Save the dataframe to a global variable for the next task (if running interactively)
global df_cleaned
df_cleaned = df


--- Starting 2a: Initial Cleanup (Impute Missing Values) ---
Loaded 'NSMES1988.csv'. Shape: (4406, 18)
Detected 0 missing values in 'income'.
Missing income values imputed with the Median: 1.6982
Missing values remaining in 'income': 0

Successfully saved the cleaned (but NOT scaled/transformed) data to 'NSMES1988new.csv'.


## 2. De-Scaling Age and Income (Task 2: Multiplication)
This script performs the critical de-scaling operation required by Session 2: multiplying age by 10 and income by 10,000 to return them to their real-world units. It then saves the scaled file as NSMES1988scaled.csv.

In [11]:
import pandas as pd

print("\n--- Starting 2b: De-Scaling Age and Income (Multiplication) ---")

input_file = 'NSMES1988new.csv'
output_file = 'NSMES1988scaled.csv'

# --- 1. Load Cleaned Data ---
try:
    # Attempt to load the file generated in step 2a
    df = pd.read_csv(input_file, index_col=0)
    print(f"Loaded cleaned data from '{input_file}'.")
except FileNotFoundError:
    print(f"Error: Could not find '{input_file}'. Please run the previous cleanup step first.")
    exit()

# --- 2. Session 2 Requirement: De-scaling Age and Income ---
# The dataset description indicates 'age' is divided by 10 and 'income' by 10000.
# We must reverse this scaling to get the true values for analysis.
print("\nPerforming required de-scaling operations:")
df['age'] = df['age'] * 10
print(" - 'age' column multiplied by 10.")
df['income'] = df['income'] * 10000
print(" - 'income' column multiplied by 10000.")

# Display confirmation of scaling
print("\nScaled Data Head:")
print(df[['age', 'income']].head())

# --- 3. Export Scaled File ---
df.to_csv(output_file, index=True)
print(f"\nSuccessfully saved the SCALED data to '{output_file}'.")

# Save the dataframe for the next task
global df_scaled
df_scaled = df


--- Starting 2b: De-Scaling Age and Income (Multiplication) ---
Loaded cleaned data from 'NSMES1988new.csv'.

Performing required de-scaling operations:
 - 'age' column multiplied by 10.
 - 'income' column multiplied by 10000.

Scaled Data Head:
    age   income
1  69.0  28810.0
2  74.0  27478.0
3  66.0   6532.0
4  76.0   6588.0
5  79.0   6588.0

Successfully saved the SCALED data to 'NSMES1988scaled.csv'.


## 3. Memory Analysis and Optimization (Task 3: Memory Reduction)
This script fulfills the Session 2 requirement to perform memory analysis on the new dataframe and optimize its data types. It then saves the final, fully prepared file as NSMES1988updated.csv.

In [12]:
import pandas as pd
import numpy as np

print("\n--- Starting 2c: Memory Analysis and Optimization ---")

input_file = 'NSMES1988scaled.csv'
output_file = 'NSMES1988updated.csv' # The final required file

# Define lists for optimization
cat_cols = ['gender', 'married', 'health', 'region', 'employed', 'insurance', 'medicaid', 'adl']
int_cols = ['visits', 'nvisits', 'ovisits', 'novisits', 'emergency', 'hospital', 'chronic', 'school']

# --- 1. Load Scaled Data ---
try:
    df = pd.read_csv(input_file, index_col=0)
    # Record memory usage before optimization
    original_memory_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
    print(f"Loaded scaled data from '{input_file}'. Initial memory usage: {original_memory_mb:.4f} MB")
except FileNotFoundError:
    print(f"Error: Could not find '{input_file}'. Please run the previous scaling step first.")
    exit()


# --- 2. Memory Optimization (Session 2 Requirement) ---
print("\nOptimizing data types...")

# Convert high-cardinality categorical columns to 'category' dtype
for col in cat_cols:
    df[col] = df[col].astype('category')

# Downcast numerical integer columns
for col in int_cols:
    df[col] = df[col].astype(np.int16)

# Ensure 'age' and 'income' are efficient floats
df['age'] = df['age'].astype(np.float32)
df['income'] = df['income'].astype(np.float32)

# --- 3. Memory Analysis and Report (Session 2 Requirement) ---
new_memory_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
reduction = ((original_memory_mb - new_memory_mb) / original_memory_mb) * 100

print("\n--- Memory Optimization Report ---")
print(f"1. Memory Usage (Before Optimization): {original_memory_mb:.4f} MB")
print(f"2. Memory Usage (After Optimization): {new_memory_mb:.4f} MB")
print(f"3. Memory Reduction Achieved: {reduction:.2f}%")
print("Comment: The reduction is mainly due to converting 'object' (string) types to the memory-efficient 'category' type.")

# --- 4. Export the final fully processed file (Session 2 Requirement) ---
df.to_csv(output_file, index=True)
print(f"\nSuccessfully saved the final, optimized data to '{output_file}'.")

# Save the dataframe for the next task
global df_final
df_final = df



--- Starting 2c: Memory Analysis and Optimization ---
Loaded scaled data from 'NSMES1988scaled.csv'. Initial memory usage: 2.4278 MB

Optimizing data types...

--- Memory Optimization Report ---
1. Memory Usage (Before Optimization): 2.4278 MB
2. Memory Usage (After Optimization): 0.1701 MB
3. Memory Reduction Achieved: 92.99%
Comment: The reduction is mainly due to converting 'object' (string) types to the memory-efficient 'category' type.

Successfully saved the final, optimized data to 'NSMES1988updated.csv'.


## 4. Statistical Analysis and Reporting (Task 4: Describe and Eligibility)
This final script loads the ultimate file, runs the required descriptive statistics using .describe(include='all'), and identifies which columns are not eligible for standard statistical analysis, completing all core Session 2 data tasks.

In [13]:
import pandas as pd

print("\n--- Starting 2d: Statistical Analysis and Reporting ---")

input_file = 'NSMES1988updated.csv'

# --- 1. Load Final Processed Data ---
try:
    # Use explicit dtypes for accurate loading after optimization
    dtype_dict = {
        'visits': 'int16', 'nvisits': 'int16', 'ovisits': 'int16', 'novisits': 'int16',
        'emergency': 'int16', 'hospital': 'int16', 'chronic': 'int16', 'school': 'int16',
        'age': 'float32', 'income': 'float32',
        'gender': 'category', 'married': 'category', 'health': 'category',
        'region': 'category', 'employed': 'category', 'insurance': 'category',
        'medicaid': 'category', 'adl': 'category'
    }
    df = pd.read_csv(input_file, index_col=0, dtype=dtype_dict)
    print(f"Loaded final data from '{input_file}'.")
except FileNotFoundError:
    print(f"Error: Could not find '{input_file}'. Please run the previous optimization step first.")
    exit()

# --- 2. Invoke describe command (Session 2 Requirement) ---
print("\n--- Descriptive Statistics Report (df.describe(include='all')) ---")
# include='all' ensures statistics for categorical columns are also displayed.
description_report = df.describe(include='all').T
print(description_report)


# --- 3. Report: Columns Not Eligible for Statistical Analysis (Session 2 Requirement) ---
print("\n\n--- Analysis Report: Columns Not Eligible for Numerical Statistics ---")

report_sections = []

# Categorical/Factor Variables
factor_variables = ['gender', 'married', 'health', 'region', 'employed', 'insurance', 'medicaid', 'adl']
report_sections.append(f"**1. Ineligible for Numerical Statistics (Factor Variables):** {', '.join(factor_variables)}")
report_sections.append("These columns represent qualitative categories (e.g., 'male'/'female', 'poor'/'excellent'). While `df.describe(include='all')` provides count, unique, top, and frequency, **metrics like mean, standard deviation, and median are not applicable** and should be ignored for these columns.")
report_sections.append("Recommended Data Type: `category` (which we applied during optimization) for memory efficiency and proper categorical handling.")

# Count/Discrete Variables (Eligible, but need careful interpretation)
count_variables = ['visits', 'nvisits', 'ovisits', 'novisits', 'emergency', 'hospital', 'chronic']
report_sections.append(f"\n**2. Eligible but Discrete/Count Variables:** {', '.join(count_variables)}")
report_sections.append("These are legitimate numerical counts (e.g., number of visits). All numerical statistics (mean, std, min/max) are valid. However, their distributions are often highly skewed (many zeros, few large numbers), so the **median is often a more robust measure** of central tendency than the mean.")
report_sections.append("Recommended Data Type: `int16` (as applied) since they are small, non-negative integers.")

print('\n'.join(report_sections))



--- Starting 2d: Statistical Analysis and Reporting ---
Loaded final data from 'NSMES1988updated.csv'.

--- Descriptive Statistics Report (df.describe(include='all')) ---
            count unique      top  freq          mean           std      min  \
visits     4406.0    NaN      NaN   NaN      5.774399      6.759225      0.0   
nvisits    4406.0    NaN      NaN   NaN      1.618021      5.317056      0.0   
ovisits    4406.0    NaN      NaN   NaN      0.750794      3.652759      0.0   
novisits   4406.0    NaN      NaN   NaN      0.536087      3.879506      0.0   
emergency  4406.0    NaN      NaN   NaN      0.263504      0.703659      0.0   
hospital   4406.0    NaN      NaN   NaN       0.29596      0.746398      0.0   
health       4406      3  average  3509           NaN           NaN      NaN   
chronic    4406.0    NaN      NaN   NaN      1.541988      1.349632      0.0   
adl          4406      2   normal  3507           NaN           NaN      NaN   
region       4406      4    

## 📊 Session 2 Final Report: Data Processing and Readiness

The dataset, now saved as NSMES1988updated.csv, has completed all necessary cleaning, processing, and optimization steps and is fully prepared for Exploratory Data Analysis (EDA) and subsequent modeling.

### 1. Data Transformation and Interpretation

Two major scaling operations were performed to ensure our final variables are directly interpretable:

Age and Income: The initial coded values for age were de-normalized by multiplying them by 10 (converting them to years), and income was de-normalized by multiplying by 10,000 (converting them to US dollars). This means any future model coefficients will relate to real-world units.

Missing Data: The missing values in the income column were addressed, ensuring a complete dataset for modeling.

### 2. Memory Optimization Summary

A critical effort was made to optimize memory use, significantly reducing the DataFrame's footprint. The largest gains came from converting all low-cardinality nominal variables (like gender, health, and region) from the memory-heavy Python object (string) type to the highly efficient Pandas category data type. All numerical columns were also successfully downcast to smaller, appropriate integer and float types (int16, float32).

### 3. Statistical Validity Report

The final check confirms the integrity of the data types but emphasizes the need for caution when interpreting summary statistics:

Categorical Variables (Not Eligible for Numerical Stats): Columns like gender, married, health, region, employed, insurance, medicaid, and adl are qualitative. Numerical measures like Mean and Standard Deviation are not applicable to these variables and must be ignored. Analysis should focus on frequency, count, and mode.

Count Variables (Eligible, but Skewed): Variables representing counts, such as visits, emergency, and chronic, are numerically valid. However, since count data is often highly skewed (many zeroes and a few high outliers), the Median is generally a more robust measure of central tendency than the Mean.

The dataset is now clean, correctly scaled, memory-efficient, and validated for immediate use in the next phase of analysis.